# 10 - TuringServe benchmark on the T4 (Build Guide Phase 8, step 10)
Benchmark the exact commit tagged `m3-rc`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must run before torch is imported

In [ ]:
!nvidia-smi
!pip uninstall -y -q torchvision torchaudio
!pip install -q torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q "transformers>=4.45,<5" safetensors fastapi "uvicorn[standard]" httpx openai \
    prometheus-client psutil hypothesis pytest pytest-asyncio pytest-timeout datasets pandas matplotlib

**Run -> Restart session**, then run the first cell again and continue below.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must run before torch is imported

In [ ]:
!git clone https://github.com/Muskanpandey0304/turingserve.git /kaggle/working/turingserve
%cd /kaggle/working/turingserve
!git checkout m3-rc
!pip install -q -e . --no-deps
!python scripts/check_versions.py

In [ ]:
!hf download Qwen/Qwen2.5-1.5B-Instruct --local-dir /tmp/models/Qwen2.5-1.5B-Instruct
!hf download Qwen/Qwen2.5-0.5B-Instruct --local-dir /tmp/models/Qwen2.5-0.5B-Instruct
!hf download anon8231489123/ShareGPT_Vicuna_unfiltered ShareGPT_V3_unfiltered_cleaned_split.json --repo-type dataset --local-dir /tmp/datasets/sharegpt

In [ ]:
os.environ["TS_MODEL_15"] = "/tmp/models/Qwen2.5-1.5B-Instruct"
os.environ["TS_MODEL_05"] = "/tmp/models/Qwen2.5-0.5B-Instruct"
os.environ["TS_SHAREGPT"] = "/tmp/datasets/sharegpt/ShareGPT_V3_unfiltered_cleaned_split.json"
os.environ["TS_KAGGLE"] = "1"

In [ ]:
import subprocess, time, requests
log = open("/kaggle/working/server.log", "w")
p = subprocess.Popen(["turingserve", "serve", "--model", os.environ["TS_MODEL_15"], "--served-model-name", "qwen", "--port", "8000"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(5)

In [ ]:
!python -m benchmarks.serve_bench --base-url http://localhost:8000 --model qwen --tokenizer $TS_MODEL_15 --dataset sharegpt --num-prompts 500 --mode concurrency --concurrency 1,4,16,32,64 --ignore-eos --label t4-1.5b --gpu-log --out /kaggle/working/results

Restart with `--max-model-len 8192` and run the prefill dataset (2,048 / 4,096 / 8,000 tokens):

In [ ]:
p.terminate(); p.wait()

In [ ]:
import subprocess, time, requests
log = open("/kaggle/working/server.log", "w")
p = subprocess.Popen(["turingserve", "serve", "--model", os.environ["TS_MODEL_15"], "--served-model-name", "qwen", "--port", "8000", "--max-model-len", "8192"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(5)

In [ ]:
!python -m benchmarks.serve_bench --base-url http://localhost:8000 --model qwen --tokenizer $TS_MODEL_15 --dataset prefill --mode concurrency --concurrency 1 --label t4-1.5b-prefill --gpu-log --out /kaggle/working/results

Then the HF baseline: `python -m benchmarks.baselines.hf_server --model $TS_MODEL_15 --port 8003` and the same serve_bench commands (ShareGPT concurrency 1, 4, 16 with 200 prompts). Download every result JSON and commit it.